# edgar-extract — LoRA fine-tune (ücretsiz T4)

Adım ②. Ayrıntılı gerekçeler: repodaki `COLAB.md`.

**Runtime → Change runtime type → T4 GPU** seçili olmalı. Hücreleri sırayla çalıştırın.

İki yerde durup çıktıya bakmanız isteniyor (5. ve 6. hücre). Oralar, üç saatlik bir
koşuyu boşa harcamamak için var.

In [ ]:
# 1) GPU DOĞRULAMA
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# Beklenen: Tesla T4, 15360 MiB, 7.5
# 7.5 = Turing = bf16 YOK. train_lora.py bunu kendisi görüp fp16 seçiyor;
# sizin bir şey yapmanız gerekmiyor. Rehberlerden kopyaladığınız BAŞKA bir
# script bf16=True (TRL varsayılanı) ile burada patlarsa sebebi budur.

In [ ]:
# 2) KURULUM — sürümler SABİT
# TRL'in SFT API'si bu proje yazılırken değişti (max_seq_length -> max_length,
# varsayılan 1024). Eski adı veren script HATA VERMEZ, sessizce her örneği keser.
!pip install -q transformers==5.14.1 trl==1.9.2 peft==0.20.0 datasets==5.0.1 accelerate==1.14.0 bitsandbytes

In [ ]:
# 3) PAKETİ YÜKLEYİN — yerelde `python src/pack_colab.py` ile üretilen
#    data/processed/colab-bundle.zip dosyasını seçin.
import os, zipfile
from google.colab import files

os.makedirs('/content/edgar-extract', exist_ok=True)
up = files.upload()
name = list(up)[0]
with zipfile.ZipFile(name) as z:
    z.extractall('/content/edgar-extract')
os.chdir('/content/edgar-extract')

# Paket TAM mı? Eksik dosyayı Colab'de fark etmek oturumu boşa harcar.
import json, pathlib
for f in ['src/prompt.py','src/train_lora.py','src/predict.py',
          'data/processed/sft_train.jsonl','data/processed/sft_dev.jsonl',
          'data/processed/sft_test.jsonl','data/processed/token_report.json']:
    print(('VAR ' if pathlib.Path(f).exists() else 'EKSİK'), f)
r = json.load(open('data/processed/token_report.json'))
print('\nölçülen taban seq_len:', r['min_seq_len_no_truncation'], '-> önerilen', r['recommended_seq_len'])

---
## 🔴 DURAK 1 — smoke test

Aşağıdaki hücre öğrenmek için değil. Baktığınız **tek satır**:

```
kayip maskesi: ... token'in ...'sinde kayip hesaplaniyor (%4.7)
```

**~%5 olmalı.** Bu, kaybın yalnız JSON hedefinde hesaplandığı anlamına gelir — 700
token'lık talimat maskeleniyor. Oran yarıdan büyükse `completion_only_loss`
çalışmıyordur ve model **talimatı üretmeyi** öğrenir. O durumda devam etmeyin.

In [ ]:
# 4) SMOKE — 135M model, 2 adım, ~2 dakika
!python src/train_lora.py --smoke

---
## 🔴 DURAK 2 — VRAM probu

Gerçek modelle birkaç adım koşup **tepe VRAM**'i ölçer. OOM'u üç saatlik koşunun
40. adımında değil, burada görün.

`--probe` kayıtları **gerçek token uzunluğuna göre** sıralayıp en uzunları öne alır.
Bu ayrım önemli: dosyalar accession'a göre sıralı, yani ilk adımlar rastgele
uzunlukta — sıralamayan bir sonda **geçer**, sonra gerçek koşu en uzun diziye
geldiğinde patlar. Tepe VRAM en uzun diziyle oluşur, o yüzden prob onunla başlar.

Çıkan sayı bir **tavan**. %85'in altındaysa gerçek koşu rahat sığar. Üstündeyse
sırasıyla: `--rank 8` → `--4bit`.
**`--max-length`'i DÜŞÜRMEYİN** — 3072 ölçülmüş taban, altına inmek örnekleri keser
ve kesilen yer tam da çıkarılacak alanların bulunduğu bölgedir.

Prob adaptörü **kaydetmez**; sadece ölçer.

In [ ]:
# 5) VRAM PROBU — en uzun dizilerle, adaptör kaydedilmez
!python src/train_lora.py --probe

In [ ]:
# 6) GERÇEK KOŞU — 99 eğitim örneği
# ---------------------------------------------------------------- YAPILANDIRMA
EPOCHS = 5      # 3 = yayınlanmış koşu · 5 = schema/EPOCH_KARARI.md denemesi
TAG    = "e5"   # 3 epoch için "" bırakın
# -----------------------------------------------------------------------------
OUT = "models/lora-qwen2.5-1.5b" + (f"-{TAG}" if TAG else "")
print("adaptör dizini:", OUT, "| epoch:", EPOCHS)

# 🔴 EPOCHS'u OUT'u değiştirmeden artırmayın. 3 epoch adaptörü, yayınlanmış
#    sayıların (%61,1) arkasındaki TEK artifact — üzerine yazılırsa geri gelmez.
#
# 🔴 Bu bir DEVAM değil, YENİ koşudur. Öğrenme oranı zamanlayıcısı toplam adım
#    sayısına göre kurulur (linear decay, transformers varsayılanı), yani 5 epoch
#    koşusundaki checkpoint-39, 3 epoch koşusundakiyle AYNI MODEL DEĞİLDİR.
#    Soru "4. ve 5. epoch iyileştirdi mi" değil: "bu koşunun dev'deki en iyi
#    checkpoint'i 17/25'i geçiyor mu".
#
# Başlangıçta basılan "optimizer adimi" satırına bakın — LoRA'nın kaç kez
# güncellendiği bu. Tahmin kabadır; gerçek adım 5 epoch için 65 (13 x 5) olmalı.
#
# 🔴 loss 'nan' olursa İLK ŞÜPHELİ fp16'dır (T4'te bf16 yok), model ya da veri
#    değil. Veriyi kurcalamadan önce --lr 5e-5 deneyin.
#
# Diğer hiperparametreler DEĞİŞMEZ (rank 16, lr 1e-4, batch 1, grad-accum 8,
# max-length 3072, seed 42). İki şeyi aynı anda değiştirmek hangisinin etki
# ettiğini ölçülemez yapar.

!python src/train_lora.py --epochs {EPOCHS} --out {OUT}

---
## Sıra önemli: önce DEV, sonra TEST

`dev` (25 kayıt) model seçimi için — epoch/lr/checkpoint burada kıyaslanır.
`test` (36 kayıt) **bir kez** bakılır; şimdi bakılırsa dondurulmuş baseline sayısı
(%27,8) anlamını kaybeder.

Bu yüzden önce yalnız dev tahminleri üretilir, indirilir, **yerelde** ölçülür
(altın etiketler pakette yok — bilerek). Yapılandırma kesinleştikten sonra test.

In [ ]:
# 7) DEV tahminleri — HER EPOCH CHECKPOINT'I AYRI (model secimi)
#
# save_strategy="epoch" her epoch'un adaptorunu birakiyor; secim onlarin
# ARASINDA yapilir. Yalniz son adaptorle tahmin uretmek "kacinci epoch daha iyi"
# sorusunu sorulamaz hale getirir.
import glob, subprocess, os

# 6. hucre kosulmadiysa (oturum koptu, yeniden baglandiniz) elle ayarlayin.
try:
    OUT, TAG
except NameError:
    OUT, TAG = "models/lora-qwen2.5-1.5b-e5", "e5"

# Dosya adina TAG giriyor: aksi halde 5 epoch kosusunun preds_ft_dev_13.jsonl'i
# 3 epoch kosusununkinin UZERINE yazar ve yayinlanmis secim kaydi kaybolur.
pre = f"preds_ft_{TAG}_dev" if TAG else "preds_ft_dev"

ckpts = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda s: int(s.split("-")[-1]))
print("adaptor dizini:", OUT)
print("bulunan checkpoint:", [os.path.basename(c) for c in ckpts] or "YOK")
assert ckpts, f"{OUT} altinda checkpoint yok — 6. hucre kosuldu mu?"

for ck in ckpts:
    n = ck.split("-")[-1]
    subprocess.run(["python", "src/predict.py", "--adapter", ck, "--split", "dev",
                    "-o", f"data/processed/{pre}_{n}.jsonl"], check=True)

# TEST'e BU KOSUDA dokunulmuyor. Karar kurali (schema/EPOCH_KARARI.md) once
# dev'de uygulanir; test ancak yapilandirma kesinlesirse ve "IKINCI olcum"
# etiketiyle kosulur.
!zip -qr dev_preds_{TAG}.zip data/processed/{pre}_*.jsonl
from google.colab import files
files.download(f'dev_preds_{TAG}.zip')

In [ ]:
# 8) TEST tahminleri — YAPILANDIRMA KESINLESTIKTEN SONRA
#
# SECILEN checkpoint'i yazin (7. adimda dev'de en iyi cikan). Bos birakirsaniz
# son adaptor kullanilir.
SECILEN = ""   # ornek: "models/lora-qwen2.5-1.5b/checkpoint-36"

adapter = SECILEN or "models/lora-qwen2.5-1.5b"
print("kullanilan adaptor:", adapter)

!python src/predict.py --adapter {adapter} --split test

# 🔴 AYNI modelin ADAPTORSUZ hali. Bu kosu opsiyonel DEGIL: fine-tuned skor tek
# basina "LoRA mi ogretti, taban model zaten biliyor muydu" sorusunu AYIRT ETMEZ.
# Ayni agirliklar, ayni talimat, adaptor yok — farki yalnizca bu olcum verir.
!python src/predict.py --split test -o data/processed/preds_base1.5b_test.jsonl

# Dorduncu yarismaci: prompted BUYUK model, ayni T4'te 4-bit.
# Sigmazsa Qwen2.5-3B-Instruct'a dusun ve raporda oyle yaz.
!python src/predict.py --model Qwen/Qwen2.5-7B-Instruct --4bit --split test -o data/processed/preds_prompted7b_test.jsonl

from google.colab import files
files.download('data/processed/preds_ft_test.jsonl')
files.download('data/processed/preds_base1.5b_test.jsonl')
files.download('data/processed/preds_prompted7b_test.jsonl')


In [ ]:
# 9) ADAPTÖRÜ İNDİRİN — artifact'ın kendisi, birkaç MB
!zip -qr adapter.zip models/lora-qwen2.5-1.5b
from google.colab import files
files.download('adapter.zip')

---
## Sonra: yerelde ölçüm

İndirilen dosyaları `data/processed/` altına koyup:

```bash
python src/evaluate.py data/processed/preds_ft_dev_*.jsonl --split dev    # seçim: en iyi epoch

python src/evaluate.py \
  data/processed/preds_regex_test.jsonl \
  data/processed/preds_ft_test.jsonl \
  data/processed/preds_prompted7b_test.jsonl \
  --json-out data/processed/eval_all.json
```

**Aşılması gereken çubuk** (kural-tabanlı, test): tam kayıt **%27,8** · zor vaka
**%81,6** · doğru abstention %90,2 · şema geçerliliği %100.

⚠️ Regex'in **dev** skoru (%68,0) kirlidir — kurallar dev oyulmadan önce tüm train
üzerinde ayarlandı. Fine-tuned modelin dev skoruyla karşılaştırmayın; karşılaştırma
yeri yalnız test.

Fine-tune bunları geçmezse sonuç **"fine-tune bu görevde kural-tabanlıyı yenmedi"**
olur ve öyle raporlanır. Ölçümün amacı kazanmak değil, öğrenmek.